# Predicting food inflation in Turkey

**Question:** Do exchange rates and global food prices help predict monthly food inflation in Turkey?

We will compare Turkish food-price changes with USD/TRY changes and global food-price changes, including related food groups.

No API key is needed for this workflow. Download these two Excel files manually and save them in `data/raw/` next to this notebook:

- **Turkish food CPI:** On the [TÜİK release page](https://veriportali.tuik.gov.tr/tr/press/58289), download the expenditure-group index table, “Harcama gruplarına göre endeks sonuçları”. Save it as `turkish_food_cpi.xlsx`. The workbook includes food categories; 2025=100 is the index's reference scale.
- **USD/TRY:** In [TCMB EVDS](https://evds3.tcmb.gov.tr/tumSeriler/2501/bie_dkdovytl), select the USD foreign-exchange buying rate, `TP.DK.USD.A.YTL`, from January 2005 through the last completed month. Export daily levels to Excel and save it as `usd_try_daily.xlsx`.

The cell below downloads the [FAO monthly CSV](https://www.fao.org/worldfoodsituation/foodpricesindex/en/) directly and loads any Turkish Excel files already saved. It does not clean, merge or model the data. Missing files are reported so we can add them and rerun the cell.

Install `requests`, `pandas` and `openpyxl` in the notebook's Python environment if needed: `pip install requests pandas openpyxl`.


In [ ]:
from pathlib import Path

import pandas as pd
import requests

raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

fao_url = (
    "https://www.fao.org/media/docs/worldfoodsituationlibraries/"
    "wfs-library/food_price_indices_data.csv?download=true&sfvrsn=523ebd2a_84"
)
response = requests.get(fao_url, timeout=90)
response.raise_for_status()
if not response.content or "text/html" in response.headers.get("Content-Type", "").lower():
    raise ValueError("FAO returned no data or a web page instead of a CSV.")
fao_path = raw_dir / "fao_food_price_indices.csv"
fao_path.write_bytes(response.content)
fao_raw = pd.read_csv(fao_path, skiprows=2, usecols=range(7))
print(f"Downloaded FAO data to {fao_path}")

turkish_raw = {}
for filename in ["turkish_food_cpi.xlsx", "usd_try_daily.xlsx"]:
    path = raw_dir / filename
    if path.is_file():
        turkish_raw[filename] = pd.read_excel(path, sheet_name=None)
        print(f"Loaded {filename}")
    else:
        print(f"Download {filename} manually into {raw_dir}, then rerun this cell.")
